# Anima + WAI-Anima — ComfyUI Colab

Один ноутбук для Anima Aesthetic v1.1 и WAI-Anima v1.0. Устанавливает ComfyUI, ComfyUI-Manager, Anima-LLLite и готовые T2I/ControlNet/Inpaint workflow. Токены берутся из Colab Secrets (`HF_TOKEN`, `CIVITAI_API_TOKEN`) или запрашиваются интерактивно.

In [ ]:
# @title 1) Tokens and paths
import os, getpass
from pathlib import Path
try:
    from google.colab import userdata
except Exception:
    userdata = None

def secret(name):
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if isinstance(value, dict):
        value = value.get('value') or value.get('token') or next(iter(value.values()), '')
    if not isinstance(value, str):
        value = str(value) if value else ''
    return value.strip() or os.environ.get(name, '').strip()

HF_TOKEN = secret('HF_TOKEN') or secret('HUGGINGFACE_TOKEN')
CIVITAI_API_TOKEN = secret('CIVITAI_API_TOKEN')
if not HF_TOKEN: HF_TOKEN = getpass.getpass('Hugging Face token (required): ').strip()
if not CIVITAI_API_TOKEN: CIVITAI_API_TOKEN = getpass.getpass('Civitai API token (recommended): ').strip()
if not HF_TOKEN: raise RuntimeError('HF_TOKEN is required.')
os.environ.update({'HF_TOKEN': HF_TOKEN, 'HUGGINGFACE_TOKEN': HF_TOKEN})
if CIVITAI_API_TOKEN: os.environ['CIVITAI_API_TOKEN'] = CIVITAI_API_TOKEN
COMFY_ROOT = Path('/content/ComfyUI'); MODEL_ROOT = COMFY_ROOT / 'models'
print('Tokens configured without displaying their values.')


In [ ]:
# @title 2) Install ComfyUI, Manager and Anima nodes
import subprocess
def run(cmd):
    print('+', cmd); subprocess.run(cmd, shell=True, check=True)
if not COMFY_ROOT.exists(): run('git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI')
run('pip install -q -r /content/ComfyUI/requirements.txt')
NODES = {'ComfyUI-Manager':'https://github.com/ltdrdata/ComfyUI-Manager.git','ComfyUI-Workflow-Models-Downloader':'https://github.com/slahiri/ComfyUI-Workflow-Models-Downloader.git','ComfyUI-Anima-LLLite':'https://github.com/kohya-ss/ComfyUI-Anima-LLLite.git','comfyui_controlnet_aux':'https://github.com/Fannovel16/comfyui_controlnet_aux.git','comfyui-lora-manager':'https://github.com/willmiao/ComfyUI-Lora-Manager.git','rgthree-comfy':'https://github.com/rgthree/rgthree-comfy.git','was-node-suite-comfyui':'https://github.com/WASasquatch/was-node-suite-comfyui.git','ComfyUI-Image-Saver':'https://github.com/alexopus/ComfyUI-Image-Saver.git'}
for folder, repo in NODES.items():
    target = COMFY_ROOT/'custom_nodes'/folder
    if not target.exists(): run(f'git clone --depth 1 {repo} {target}')
    req = target/'requirements.txt'
    if req.exists(): run(f'pip install -q -r {req}')
print('ComfyUI and Anima node set are ready.')

In [ ]:
# @title 3) Download both Anima checkpoints and dependencies
import requests
def download(url, target, headers=None):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>1024: print('exists:',target); return
    h={'Authorization':f'Bearer {HF_TOKEN}'} if 'huggingface.co' in url else {}
    if headers: h.update(headers)
    with requests.get(url,headers=h,stream=True,timeout=60) as r:
        r.raise_for_status()
        with open(target,'wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk: f.write(chunk)
    print('downloaded:',target)
def civitai(url,target):
    h={'Authorization':f'Bearer {CIVITAI_API_TOKEN}'} if CIVITAI_API_TOKEN else {}
    download(url,target,h)
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors',MODEL_ROOT/'text_encoders/qwen_3_06b_base.safetensors')
download('https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors',MODEL_ROOT/'vae/qwen_image_vae.safetensors')
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-any-test-like-v2.safetensors',MODEL_ROOT/'model_patches/anima-lllite-any-test-like-v2.safetensors')
download('https://huggingface.co/Comfy-Org/Anima-LLLite/resolve/main/model_patches/anima-lllite-inpainting-v2.safetensors',MODEL_ROOT/'model_patches/anima-lllite-inpainting-v2.safetensors')
civitai('https://civitai.red/api/download/models/3126581?fileId=3007030',MODEL_ROOT/'diffusion_models/anima/anima_aestheticV11.safetensors')
civitai('https://civitai.red/api/download/models/2983680?fileId=2863158',MODEL_ROOT/'diffusion_models/anima/waiANIMA_v10Base10.safetensors')
print('Both checkpoints are available in ComfyUI.')

In [ ]:
# @title 5) Launch ComfyUI + self-healing LocalTunnel
import base64, os, queue, re, shutil, socket, subprocess, threading, time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests

LOW_VRAM_STABLE = False  # True: slower but safer for large images on free Colab
COMFY_ROOT = Path(globals().get('COMFY_ROOT', '/content/ComfyUI'))
OUTPUT_DIR = COMFY_ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def stop_process(proc):
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()


# Safe rerun: stop processes created by every older launch-cell version.
old_stop = globals().get('_TUNNEL_STOP')
if old_stop is not None:
    old_stop.set()
for process_name in ('_TUNNEL_PROC', '_COMFY_PROC', 'tunnel', 'comfy'):
    stop_process(globals().get(process_name))
old_log = globals().get('_COMFY_LOG')
if old_log is not None:
    try:
        old_log.close()
    except Exception:
        pass


def ensure_localtunnel():
    if shutil.which('lt') is not None:
        return
    if shutil.which('npm') is None:
        raise RuntimeError('npm is missing; LocalTunnel cannot be installed.')
    subprocess.run(
        ['npm', 'install', '--global', 'localtunnel@2.0.2'],
        check=True,
    )
    if shutil.which('lt') is None:
        raise RuntimeError('LocalTunnel installation finished, but lt was not found.')


ensure_localtunnel()

comfy_args = [
    'python', 'main.py', '--listen', '0.0.0.0', '--port', '8188',
    '--enable-cors-header', '*', '--output-directory', str(OUTPUT_DIR),
]
comfy_args += (
    ['--novram', '--disable-smart-memory', '--cache-none', '--force-upcast-attention']
    if LOW_VRAM_STABLE else ['--lowvram', '--preview-method', 'auto']
)
_COMFY_LOG = open('/content/comfyui.log', 'a', encoding='utf-8', buffering=1)
_COMFY_PROC = subprocess.Popen(
    comfy_args, cwd=COMFY_ROOT, stdout=_COMFY_LOG, stderr=subprocess.STDOUT,
)


def local_comfy_ready(timeout=240):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if _COMFY_PROC.poll() is not None:
            raise RuntimeError('ComfyUI exited. Inspect /content/comfyui.log')
        try:
            with socket.create_connection(('127.0.0.1', 8188), timeout=2):
                response = requests.get('http://127.0.0.1:8188/system_stats', timeout=5)
                if response.ok:
                    return True
        except (OSError, requests.RequestException):
            time.sleep(2)
    return False


if not local_comfy_ready():
    raise TimeoutError('ComfyUI did not become ready within 240 seconds.')
print('ComfyUI is ready locally. Output:', OUTPUT_DIR)


def read_process_lines(proc, lines):
    for line in iter(proc.stdout.readline, ''):
        lines.put(line.rstrip())


def start_localtunnel(timeout=90):
    proc = subprocess.Popen(
        ['lt', '--port', '8188', '--local-host', '127.0.0.1'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = queue.Queue()
    threading.Thread(target=read_process_lines, args=(proc, lines), daemon=True).start()
    recent = []
    deadline = time.time() + timeout
    while time.time() < deadline and proc.poll() is None:
        try:
            line = lines.get(timeout=1)
        except queue.Empty:
            continue
        recent = (recent + [line])[-10:]
        match = re.search(r'https://[-a-z0-9]+\.loca\.lt', line, re.I)
        if match:
            return proc, match.group(0), recent
    stop_process(proc)
    return None, None, recent


TUNNEL_HEADERS = {
    'Bypass-Tunnel-Reminder': 'true',
    'User-Agent': 'comfy-colab-healthcheck',
}


def public_comfy_ready(url, timeout=12):
    try:
        response = requests.get(
            url.rstrip('/') + '/system_stats',
            headers=TUNNEL_HEADERS,
            timeout=timeout,
        )
        if not response.ok:
            return False
        payload = response.json()
        return isinstance(payload, dict) and ('system' in payload or 'devices' in payload)
    except (requests.RequestException, ValueError):
        return False



def local_backend_responding(timeout=5):
    try:
        return requests.get('http://127.0.0.1:8188/system_stats', timeout=timeout).ok
    except requests.RequestException:
        return False


def public_tunnel_responding(url, timeout=15):
    try:
        response = requests.get(
            url.rstrip('/') + '/',
            headers=TUNNEL_HEADERS,
            timeout=timeout,
        )
        return response.ok and '<html' in response.text[:4096].lower()
    except requests.RequestException:
        return False


def public_frontend_assets_ready(url):
    # Check entry JS/CSS and CSS-linked fonts/icons before exposing the URL.
    session = requests.Session()
    session.headers.update(TUNNEL_HEADERS)
    try:
        root_response = session.get(url.rstrip('/') + '/', timeout=20)
        if not root_response.ok:
            return False
        references = re.findall(
            r'''(?:src|href)=["']([^"']+)["']''',
            root_response.text,
            flags=re.I,
        )
        checked = set()
        css_bodies = []
        for reference in references:
            asset_url = urljoin(root_response.url, reference)
            if urlparse(asset_url).netloc != urlparse(root_response.url).netloc:
                continue
            if asset_url in checked:
                continue
            checked.add(asset_url)
            response = session.get(asset_url, timeout=20)
            if not response.ok:
                return False
            if urlparse(asset_url).path.lower().endswith('.css'):
                css_bodies.append((asset_url, response.text))
        for css_url, css_body in css_bodies:
            for reference in re.findall(r'''url\(["']?([^"')]+)''', css_body, flags=re.I):
                if reference.startswith('data:'):
                    continue
                asset_url = urljoin(css_url, reference)
                if urlparse(asset_url).netloc != urlparse(root_response.url).netloc:
                    continue
                if asset_url in checked:
                    continue
                checked.add(asset_url)
                if not session.get(asset_url, timeout=20).ok:
                    return False
        return True
    except requests.RequestException:
        return False

def public_image_route_ready(url):
    # Verify the exact /view route used by Save Image previews.
    probe_name = '_localtunnel_image_probe.png'
    probe_path = OUTPUT_DIR / probe_name
    probe_png = base64.b64decode(
        'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNk+A8AAQUBAScY42YAAAAASUVORK5CYII='
    )
    try:
        probe_path.write_bytes(probe_png)
        response = requests.get(
            url.rstrip('/') + '/view',
            params={'filename': probe_name, 'type': 'output', 'subfolder': ''},
            headers=TUNNEL_HEADERS,
            timeout=15,
        )
        return response.ok and response.content.startswith(b'\x89PNG')
    except requests.RequestException:
        return False
    finally:
        probe_path.unlink(missing_ok=True)


def tunnel_password():
    try:
        response = requests.get(
            'https://loca.lt/mytunnelpassword',
            headers={'User-Agent': 'curl/8'},
            timeout=15,
        )
        response.raise_for_status()
        return response.text.strip()
    except requests.RequestException:
        return '(could not fetch automatically)'


_TUNNEL_STOP = threading.Event()
_TUNNEL_PROC = None


def tunnel_supervisor():
    global _TUNNEL_PROC
    while not _TUNNEL_STOP.is_set():
        print('Starting LocalTunnel...')
        proc, url, recent = start_localtunnel()
        if proc is None:
            print('LocalTunnel failed:', recent[-4:] or ['no output'])
            _TUNNEL_STOP.wait(10)
            continue

        _TUNNEL_PROC = proc
        if not (
            public_comfy_ready(url)
            and public_image_route_ready(url)
            and public_frontend_assets_ready(url)
        ):
            print('LocalTunnel failed /system_stats, /view, or frontend asset verification; restarting...')
            stop_process(proc)
            _TUNNEL_STOP.wait(5)
            continue

        print('Open ComfyUI:', url)
        print('Tunnel password (only if the consent page asks):', tunnel_password())
        print('LocalTunnel verified: frontend assets, /system_stats, and Save Image /view are reachable.')

        public_failure_since = None
        local_busy_reported = False
        while not _TUNNEL_STOP.wait(5):
            if _COMFY_PROC.poll() is not None:
                print('ComfyUI stopped. Inspect /content/comfyui.log')
                stop_process(proc)
                return
            if proc.poll() is not None:
                print('LocalTunnel process exited; restarting immediately...')
                break
            if public_tunnel_responding(url):
                public_failure_since = None
                local_busy_reported = False
                continue
            if not local_backend_responding():
                # Sampling may temporarily delay ComfyUI HTTP responses. Keep the same URL.
                public_failure_since = None
                if not local_busy_reported:
                    print('ComfyUI is busy locally; keeping the current tunnel URL.')
                    local_busy_reported = True
                continue
            local_busy_reported = False
            if public_failure_since is None:
                public_failure_since = time.time()
                print('LocalTunnel is temporarily unhealthy; waiting for its built-in reconnect...')
            elif time.time() - public_failure_since >= 30:
                print('LocalTunnel stayed unhealthy for 30 seconds; recreating it...')
                stop_process(proc)
                break
        stop_process(proc)


print('LocalTunnel supervisor is running. Stop this cell to close ComfyUI.')
try:
    tunnel_supervisor()
except KeyboardInterrupt:
    print('Stopping LocalTunnel and ComfyUI...')
finally:
    _TUNNEL_STOP.set()
    stop_process(_TUNNEL_PROC)
    stop_process(_COMFY_PROC)
    try:
        _COMFY_LOG.close()
    except Exception:
        pass
